In [ ]:
# Import standard and third-party libraries for data handling, modeling, and tokenization
import os
import json
import regex as re
import requests
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
import pickle
import math
import time
from collections import defaultdict
import tiktoken

import warnings
warnings.filterwarnings("ignore")
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

print(f"Using device: {DEVICE}")

# 1) Tokenization (Byte Pair Encoding)

The GPT-2 and RoBERTa tokenizers (which are pretty similar) have a clever way to deal with this: they don’t look at words as being written with Unicode characters, but with bytes. This way the base vocabulary has a small size (256), but every character you can think of will still be included and not end up being converted to the unknown token. This trick is called byte-level BPE.

[Description](https://huggingface.co/course/chapter6/5?fw=pt) | [Code](https://github.com/karpathy/minGPT/blob/master/mingpt/bpe.py)

## 1.1 BPE from Scratch

Unlike WordPiece, it's impossible to get an unknown character when using byte-level BPE

In [ ]:
# Import libraries for BPE tokenization
from transformers import AutoTokenizer
from collections import defaultdict

# Load the pretrained GPT-2 tokenizer (used only for pre-tokenization)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Define a small corpus for demonstrating BPE vocabulary construction
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

In [ ]:
# Pre-tokenize corpus: split each sentence into words using GPT-2 pre-tokenizer
# word_freqs: dict mapping word → frequency count
word_freqs = defaultdict(int)
for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(
        text
    )
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

# Build character-level alphabet from all unique characters in the corpus
# alphabet: sorted list of unique characters (initial BPE vocabulary symbols)
alphabet = []
for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()

# Initialise splits: each word is represented as a list of individual characters
# splits: dict mapping word → [char, char, ...]  (will be merged iteratively)
splits = {word: [c for c in word] for word in word_freqs.keys()}

defaultdict(<class 'int'>, {'This': 3, 'Ġis': 2, 'Ġthe': 1, 'ĠHugging': 1, 'ĠFace': 1, 'ĠCourse': 1, '.': 4, 'Ġchapter': 1, 'Ġabout': 1, 'Ġtokenization': 1, 'Ġsection': 1, 'Ġshows': 1, 'Ġseveral': 1, 'Ġtokenizer': 1, 'Ġalgorithms': 1, 'Hopefully': 1, ',': 1, 'Ġyou': 1, 'Ġwill': 1, 'Ġbe': 1, 'Ġable': 1, 'Ġto': 1, 'Ġunderstand': 1, 'Ġhow': 1, 'Ġthey': 1, 'Ġare': 1, 'Ġtrained': 1, 'Ġand': 1, 'Ġgenerate': 1, 'Ġtokens': 1})
[',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ']
{'This': ['T', 'h', 'i', 's'], 'Ġis': ['Ġ', 'i', 's'], 'Ġthe': ['Ġ', 't', 'h', 'e'], 'ĠHugging': ['Ġ', 'H', 'u', 'g', 'g', 'i', 'n', 'g'], 'ĠFace': ['Ġ', 'F', 'a', 'c', 'e'], 'ĠCourse': ['Ġ', 'C', 'o', 'u', 'r', 's', 'e'], '.': ['.'], 'Ġchapter': ['Ġ', 'c', 'h', 'a', 'p', 't', 'e', 'r'], 'Ġabout': ['Ġ', 'a', 'b', 'o', 'u', 't'], 'Ġtokenization': ['Ġ', 't', 'o', 'k', 'e', 'n', 'i', 'z', 'a', 't', 'i', 'o', 'n'], 'Ġsection':

In [ ]:
# Utility: count frequency of every adjacent (subword_a, subword_b) pair across the corpus
# Returns: dict mapping (str, str) pair → int frequency
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs


# Compute initial pair frequencies over the character-level splits
pair_freqs = compute_pair_freqs(splits)

defaultdict(<class 'int'>, {('T', 'h'): 3, ('h', 'i'): 3, ('i', 's'): 5, ('Ġ', 'i'): 2, ('Ġ', 't'): 7, ('t', 'h'): 3, ('h', 'e'): 2, ('Ġ', 'H'): 1, ('H', 'u'): 1, ('u', 'g'): 1, ('g', 'g'): 1, ('g', 'i'): 1, ('i', 'n'): 2, ('n', 'g'): 1, ('Ġ', 'F'): 1, ('F', 'a'): 1, ('a', 'c'): 1, ('c', 'e'): 1, ('Ġ', 'C'): 1, ('C', 'o'): 1, ('o', 'u'): 3, ('u', 'r'): 1, ('r', 's'): 2, ('s', 'e'): 3, ('Ġ', 'c'): 1, ('c', 'h'): 1, ('h', 'a'): 1, ('a', 'p'): 1, ('p', 't'): 1, ('t', 'e'): 2, ('e', 'r'): 5, ('Ġ', 'a'): 5, ('a', 'b'): 2, ('b', 'o'): 1, ('u', 't'): 1, ('t', 'o'): 4, ('o', 'k'): 3, ('k', 'e'): 3, ('e', 'n'): 4, ('n', 'i'): 2, ('i', 'z'): 2, ('z', 'a'): 1, ('a', 't'): 2, ('t', 'i'): 2, ('i', 'o'): 2, ('o', 'n'): 2, ('Ġ', 's'): 3, ('e', 'c'): 1, ('c', 't'): 1, ('s', 'h'): 1, ('h', 'o'): 2, ('o', 'w'): 2, ('w', 's'): 1, ('e', 'v'): 1, ('v', 'e'): 1, ('r', 'a'): 3, ('a', 'l'): 2, ('z', 'e'): 1, ('l', 'g'): 1, ('g', 'o'): 1, ('o', 'r'): 1, ('r', 'i'): 1, ('i', 't'): 1, ('h', 'm'): 1, ('m', 's'): 

In [ ]:
# Find the most frequent adjacent pair in the current splits
best_pair = ""
max_freq = None
for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

# Initialise the BPE vocabulary with a special end-of-text token and the character alphabet
# vocab: list[str] — grows with each merge step
vocab = ["<|endoftext|>"] + alphabet.copy()

# Seed the merge table with the first (highest-frequency) merge: ('Ġ', 't') → 'Ġt'
# merges: dict mapping (str, str) → merged_str
merges = {("Ġ", "t"): "Ġt"}

# Add the first merged token to the vocabulary
vocab.append("Ġt")

('Ġ', 't') 7


In [ ]:
# Utility: apply a single merge (a, b) → (a+b) to all word splits
# Modifies splits in-place and returns the updated dict
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue
        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                # Replace the adjacent pair with the merged token
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits


# Apply the first merge ('Ġ', 't') → 'Ġt' to the character-level splits
splits = merge_pair("Ġ", "t", splits)

['Ġt', 'r', 'a', 'i', 'n', 'e', 'd']


In [ ]:
# Set the target vocabulary size for BPE training
vocab_size = 50

# BPE training loop: iteratively merge the most frequent pair until vocab_size is reached
while len(vocab) < vocab_size:
    # Recompute pair frequencies after the last merge
    pair_freqs = compute_pair_freqs(splits)

    # Find the most frequent adjacent subword pair
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq

    # Apply the merge to all splits
    splits = merge_pair(*best_pair, splits)

    # Record the merge rule and extend the vocabulary
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

{('Ġ', 't'): 'Ġt', ('i', 's'): 'is', ('e', 'r'): 'er', ('Ġ', 'a'): 'Ġa', ('Ġt', 'o'): 'Ġto', ('e', 'n'): 'en', ('T', 'h'): 'Th', ('Th', 'is'): 'This', ('o', 'u'): 'ou', ('s', 'e'): 'se', ('Ġto', 'k'): 'Ġtok', ('Ġtok', 'en'): 'Ġtoken', ('n', 'd'): 'nd', ('Ġ', 'is'): 'Ġis', ('Ġt', 'h'): 'Ġth', ('Ġth', 'e'): 'Ġthe', ('i', 'n'): 'in', ('Ġa', 'b'): 'Ġab', ('Ġtoken', 'i'): 'Ġtokeni'}
['<|endoftext|>', ',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ', 'Ġt', 'is', 'er', 'Ġa', 'Ġto', 'en', 'Th', 'This', 'ou', 'se', 'Ġtok', 'Ġtoken', 'nd', 'Ġis', 'Ġth', 'Ġthe', 'in', 'Ġab', 'Ġtokeni']


In [ ]:
# Tokenize new text using the learned BPE merge rules
# Returns: list[str] — sequence of BPE subword tokens
def tokenize(text):
    # Step 1: pre-tokenize text into words (same pre-tokenizer as GPT-2)
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]

    # Step 2: split each pre-token into individual characters
    splits = [[l for l in word] for word in pre_tokenized_text]

    # Step 3: apply learned merge rules in order to build subword tokens
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    # Step 4: flatten list-of-lists into a single token sequence
    return sum(splits, [])


# Tokenize an example sentence using the learned BPE vocabulary
tokenize("This is not a token.")

['This', 'Ġis', 'Ġ', 'n', 'o', 't', 'Ġa', 'Ġtoken', '.']

# 2) Dataset

In [ ]:
# Download the Tiny Shakespeare dataset if it does not already exist locally
data_dir = os.path.join("data", "tinyshakespeare")
input_file_path = os.path.join(data_dir, "input.txt")
if not os.path.exists(input_file_path):
    data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    os.makedirs(data_dir)
    with open(input_file_path, "w") as f:
        f.write(requests.get(data_url).text)

# Read raw text and split into 90% train / 10% validation
with open(input_file_path, "r") as f:
    data = f.read()
n = len(data)
train_data = data[: int(n * 0.9)]
val_data = data[int(n * 0.9) :]

# Encode text with GPT-2 BPE tokenizer (tiktoken)
# train_ids / val_ids: list[int] — integer token IDs in range [0, vocab_size)
enc = tiktoken.get_encoding("gpt2")
train_ids = enc.encode_ordinary(train_data)
val_ids = enc.encode_ordinary(val_data)
print(f"train has {len(train_ids):,} tokens")
print(f"val has {len(val_ids):,} tokens")

# Persist token ID arrays as uint16 binary files for efficient memory-mapped loading
# train_ids shape: (num_train_tokens,) | val_ids shape: (num_val_tokens,)
train_ids = np.array(train_ids, dtype=np.uint16)
val_ids = np.array(val_ids, dtype=np.uint16)
train_ids.tofile(os.path.join(data_dir, "train.bin"))
val_ids.tofile(os.path.join(data_dir, "val.bin"))

train has 301,966 tokens
val has 36,059 tokens


In [ ]:
# Base configuration class: stores model and training hyperparameters as attributes
class GPTConfig:
    def __init__(self, vocab_size, **kwargs):
        self.vocab_size = vocab_size
        for key, value in kwargs.items():
            setattr(self, key, value)


# Concrete configuration for the nanoGPT experiment
class CustomConfig(GPTConfig):
    # ── Architecture hyperparameters ─────────────────────────────────────
    n_layer = 8  # num_layers: number of stacked transformer decoder blocks
    n_head = 8  # num_heads: number of parallel attention heads per block
    n_embd = 256  # embed_dim: token + positional embedding dimension
    embd_pdrop = 0.1  # dropout rate applied to the summed embeddings
    resid_pdrop = 0.1  # dropout rate on residual (post-attention / post-MLP) paths
    attn_pdrop = 0.1  # dropout rate on attention weight matrix
    dropout = 0.1  # general dropout probability

    # ── Data / device settings ───────────────────────────────────────────
    device = "cuda"  # target device ('cuda' or 'cpu')
    num_workers = 0  # DataLoader worker processes

    # ── Optimiser / training schedule ────────────────────────────────────
    max_iters = 2e4  # total training iterations
    batch_size = 4  # batch_num: sequences per forward pass
    block_size = 64  # max_len / seq_len: context window length
    learning_rate = 6e-4  # AdamW base learning rate
    betas = (0.9, 0.95)  # AdamW (beta1, beta2)
    weight_decay = 1e-1  # L2 regularisation coefficient
    grad_norm_clip = 1.0  # gradient norm clipping threshold


# Instantiate config; vocab_size derived from the number of training tokens
vocab_size = len(train_ids)
config = CustomConfig(vocab_size=vocab_size)

In [ ]:
# Memory-map train/val binary files for efficient random access (avoids loading all data into RAM)
# Shape: (num_tokens,)  dtype: uint16
data_dir = os.path.join("data", "tinyshakespeare")
train_data = np.memmap(os.path.join(data_dir, "train.bin"), dtype=np.uint16, mode="r")
val_data = np.memmap(os.path.join(data_dir, "val.bin"), dtype=np.uint16, mode="r")


# Dataset class: yields (input_ids, target_ids) pairs of length seq_len (block_size)
class ShakespeareDataset(Dataset):
    def __init__(self, split, block_size=128, device_type="cuda"):
        assert split in {"train", "test"}
        self.split = split
        self.block_size = block_size
        self.device_type = device_type
        # Select the correct memory-mapped array
        self.data = train_data if split == "train" else val_data

    def __len__(self):
        # Number of valid starting positions for a seq_len-length window
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Slice a contiguous block of token IDs
        # x shape: (seq_len,)  — input token IDs
        x = torch.from_numpy(self.data[idx : idx + self.block_size].astype(np.int64))
        # y shape: (seq_len,)  — target token IDs (shifted right by 1)
        y = torch.from_numpy(
            self.data[idx + 1 : idx + 1 + self.block_size].astype(np.int64)
        )

        # Move tensors to the target device
        if self.device_type == "cuda":
            # Pin memory for faster host-to-device transfer
            x, y = x.pin_memory().to("cuda", non_blocking=True), y.pin_memory().to(
                "cuda", non_blocking=True
            )
        else:
            x, y = x.to("cpu"), y.to("cpu")
        return x, y


# Instantiate datasets and DataLoaders
# Batch shape from DataLoader: x → (batch_num, seq_len), y → (batch_num, seq_len)
train_dataset = ShakespeareDataset("train", config.block_size, config.device)
train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    drop_last=False,
)
test_dataset = ShakespeareDataset("test", config.block_size, config.device)
test_loader = DataLoader(
    test_dataset,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    drop_last=False,
)

# Fetch a sample batch for shape verification
# x: (batch_num, seq_len)  y: (batch_num, seq_len)
sample_data = next(iter(train_loader))
x, y = sample_data

x: torch.Size([4, 64])
y: torch.Size([4, 64])


# 3) Modeling

<img src='https://miro.medium.com/v2/resize:fit:1400/0*T-qaxdsUriM5Z5A-' width=800>

In [ ]:
# ── Activation function ─────────────────────────────────────────────────
# NewGELU: element-wise GELU activation (identical to OpenAI GPT implementation)
# Input/Output: any shape — activation is applied element-wise, shape is preserved
class NewGELU(nn.Module):
    """
    Implementation of the GELU activation function currently in Google BERT repo (identical to OpenAI GPT).
    Reference: Gaussian Error Linear Units (GELU) paper: https://arxiv.org/abs/1606.08415
    """

    def forward(self, x):
        return (
            0.5
            * x
            * (
                1.0
                + torch.tanh(
                    math.sqrt(2.0 / math.pi) * (x + 0.044715 * torch.pow(x, 3.0))
                )
            )
        )


# ── Causal (masked) multi-head self-attention ────────────────────────────
class CausalSelfAttention(nn.Module):
    """
    A vanilla multi-head masked self-attention layer with a projection at the end.
    It's important in decoder block to have diagonal mask
    It is also possible to use torch.nn.MultiheadAttention.
    """

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # Fused QKV projection: (embed_dim) → (embed_dim * 3)
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # Output projection: (embed_dim) → (embed_dim)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        # Regularisation
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)
        self.dropout = config.dropout
        self.n_head = config.n_head
        self.n_embd = config.n_embd

        # Use Flash Attention when available (PyTorch >= 2.0) for speed
        self.flash = hasattr(torch.nn.functional, "scaled_dot_product_attention")
        if not self.flash:
            print(
                "WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0"
            )
            # Causal mask: lower-triangular ones of shape (1, 1, max_len, max_len)
            # Ensures position i can only attend to positions 0..i (autoregressive)
            self.register_buffer(
                "mask",
                torch.tril(torch.ones(config.block_size, config.block_size)).view(
                    1, 1, config.block_size, config.block_size
                ),
            )

    def forward(self, x):
        # Input: (batch_num, seq_len, embed_dim)
        B, T, C = x.size()

        # Compute Q, K, V via fused linear projection then split along last dim
        # Input: (batch_num, seq_len, embed_dim)
        # → fused: (batch_num, seq_len, embed_dim * 3)
        # → q, k, v each: (batch_num, seq_len, embed_dim)
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)

        # Reshape and transpose to separate heads
        # Input: (batch_num, seq_len, embed_dim)
        # → Output: (batch_num, num_heads, seq_len, embed_dim // num_heads)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # Compute scaled dot-product attention with causal mask
        if self.flash:
            # Flash Attention path
            # Input q/k/v: (batch_num, num_heads, seq_len, embed_dim // num_heads)
            # Output y:     (batch_num, num_heads, seq_len, embed_dim // num_heads)
            y = torch.nn.functional.scaled_dot_product_attention(
                q,
                k,
                v,
                attn_mask=None,
                dropout_p=self.dropout if self.training else 0,
                is_causal=True,
            )
        else:
            # Manual attention path
            # Q @ K^T: (batch_num, num_heads, seq_len, seq_len)  scaled by 1/sqrt(head_dim)
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            # Apply causal mask: set future positions to -inf before softmax
            # Shape unchanged: (batch_num, num_heads, seq_len, seq_len)
            att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
            # Softmax over key dimension → attention weights
            # Shape: (batch_num, num_heads, seq_len, seq_len)
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)

            # Weighted sum over values
            # (batch_num, num_heads, seq_len, seq_len) @ (batch_num, num_heads, seq_len, embed_dim // num_heads)
            # → Output: (batch_num, num_heads, seq_len, embed_dim // num_heads)
            y = att @ v

        # Re-assemble all head outputs: transpose back and merge head dimension
        # Input:  (batch_num, num_heads, seq_len, embed_dim // num_heads)
        # Output: (batch_num, seq_len, embed_dim)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Output projection + residual dropout
        # (batch_num, seq_len, embed_dim)
        y = self.resid_dropout(self.c_proj(y))
        return y


# ── Transformer decoder block ────────────────────────────────────────────
class Block(nn.Module):
    """GPT only contain decode block"""

    def __init__(self, config):
        super().__init__()
        # Pre-norm for attention sub-layer
        self.ln_1 = nn.LayerNorm(config.n_embd)
        # Causal multi-head self-attention
        self.attn = CausalSelfAttention(config)
        # Pre-norm for feed-forward sub-layer
        self.ln_2 = nn.LayerNorm(config.n_embd)

        # Position-wise feed-forward network (MLP)
        # c_fc:   (embed_dim) → (embed_dim * 4)   expand
        # act:    NewGELU activation, shape unchanged
        # c_proj: (embed_dim * 4) → (embed_dim)   project back
        self.mlp = nn.ModuleDict(
            dict(
                c_fc=nn.Linear(config.n_embd, 4 * config.n_embd),
                act=NewGELU(),
                c_proj=nn.Linear(4 * config.n_embd, config.n_embd),
                dropout=nn.Dropout(config.resid_pdrop),
            )
        )
        m = self.mlp
        # Compose MLP sub-layers into a single callable
        # (batch_num, seq_len, embed_dim)
        self.mlpf = lambda x: m.dropout(m.c_proj(m.act(m.c_fc(x))))

    def forward(self, x):
        # Input: (batch_num, seq_len, embed_dim)

        # Sub-layer 1: LayerNorm → CausalSelfAttention → residual add
        # (batch_num, seq_len, embed_dim)
        x = x + self.attn(self.ln_1(x))

        # Sub-layer 2: LayerNorm → MLP → residual add
        # (batch_num, seq_len, embed_dim)
        x = x + self.mlpf(self.ln_2(x))

        # Output: (batch_num, seq_len, embed_dim)
        return x


# ── Smoke-test Block with a sample batch ────────────────────────────────
# Token embedding table: (vocab_size, embed_dim)
wte = nn.Embedding(config.vocab_size, config.n_embd).to(config.device)
block = Block(config).to(config.device)

# Embed token IDs: Input (batch_num, seq_len) → Output (batch_num, seq_len, embed_dim)
tok_emb = wte(x)

# Forward pass through one Block
# (batch_num, seq_len, embed_dim)
block_out = block(tok_emb)

Token Embedding Size: torch.Size([4, 64, 256])
Block Output Size: torch.Size([4, 64, 256])


In [ ]:
# ── Full GPT Language Model ──────────────────────────────────────────────
class GPT(nn.Module):
    """GPT Language Model"""

    def __init__(self, config):
        super().__init__()
        self.block_size = config.block_size

        # Compute sinusoidal positional encoding matrix
        # Shape: (max_len, embed_dim) — added to token embeddings at forward time
        self.positional_encoding = self._get_positional_encoding(
            config.block_size, config.n_embd
        )
        self.transformer = nn.ModuleDict(
            dict(
                # Token embedding table: (vocab_size, embed_dim)
                # wte = nn.Embedding(config.vocab_size, config.n_embd),  [commented out — set below]
                # Learned positional embedding table (alternative): (max_len, embed_dim)
                # wpe = nn.Embedding(config.block_size, config.n_embd),  [commented out]
                # Embedding dropout
                drop=nn.Dropout(config.embd_pdrop),
                # Stack of num_layers transformer decoder blocks
                h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                # Final layer normalisation before LM head
                ln_f=nn.LayerNorm(config.n_embd),
            )
        )
        # LM head: projects embed_dim → vocab_size (weight-tied with wte)
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, vocab_size)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Token embedding table (weight-tied to lm_head)
        # Shape: (vocab_size, embed_dim)
        self.transformer.wte = nn.Embedding(config.vocab_size, config.n_embd)

        # Initialise all weights; apply scaled init to residual projections (GPT-2 paper)
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                torch.nn.init.normal_(
                    p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer)
                )

        # Report parameter count (transformer body only, excluding lm_head)
        n_params = sum(p.numel() for p in self.transformer.parameters())
        print("number of parameters: %.2fM" % (n_params / 1e6,))

    # ── Weight initialisation ────────────────────────────────────────────
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    # ── Sinusoidal positional encoding ──────────────────────────────────
    def _get_positional_encoding(self, max_seq_len, embed_size):
        # Output shape: (max_len, embed_dim)
        positional_encoding = torch.zeros(max_seq_len, embed_size)

        # position: (max_len, 1)  — integer positions 0 .. max_len-1
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()

        # div_term: (embed_dim // 2,)  — frequency scaling factors
        div_term = torch.exp(
            torch.arange(0, embed_size, 2).float() * -(math.log(10000.0) / embed_size)
        )

        # Fill even indices with sine, odd indices with cosine
        # positional_encoding[:, 0::2] shape: (max_len, embed_dim // 2)
        positional_encoding[:, 0::2] = torch.sin(position * div_term)
        positional_encoding[:, 1::2] = torch.cos(position * div_term)
        return positional_encoding

    # ── Optimiser configuration ──────────────────────────────────────────
    def configure_optimizers(self, train_config):
        # Separate parameters into weight-decayed (Linear weights) and non-decayed groups
        decay = set()
        no_decay = set()
        whitelist_weight_modules = (torch.nn.Linear,)
        blacklist_weight_modules = (torch.nn.LayerNorm, torch.nn.Embedding)
        for mn, m in self.named_modules():
            for pn, p in m.named_parameters():
                fpn = "%s.%s" % (mn, pn) if mn else pn
                if pn.endswith("bias"):
                    no_decay.add(fpn)
                elif pn.endswith("weight") and isinstance(m, whitelist_weight_modules):
                    decay.add(fpn)
                elif pn.endswith("weight") and isinstance(m, blacklist_weight_modules):
                    no_decay.add(fpn)

        # Validate every parameter is assigned to exactly one group
        param_dict = {pn: p for pn, p in self.named_parameters()}
        inter_params = decay & no_decay
        union_params = decay | no_decay

        # Build AdamW optimiser with two parameter groups
        optim_groups = [
            {
                "params": [param_dict[pn] for pn in sorted(list(decay))],
                "weight_decay": train_config.weight_decay,
            },
            {
                "params": [param_dict[pn] for pn in sorted(list(no_decay))],
                "weight_decay": 0.0,
            },
        ]
        optimizer = torch.optim.AdamW(
            optim_groups, lr=train_config.learning_rate, betas=train_config.betas
        )
        return optimizer

    # ── Forward pass ─────────────────────────────────────────────────────
    def forward(self, idx, targets=None):
        device = idx.device
        # Input idx: (batch_num, seq_len)  — integer token IDs
        b, t = idx.size()
        assert (
            t <= self.block_size
        ), f"Cannot forward sequence of length {t}, block size is only {self.block_size}"

        # Embed input tokens
        # (batch_num, seq_len) → (batch_num, seq_len, embed_dim)
        tok_emb = self.transformer.wte(idx)

        # Retrieve sinusoidal positional encoding for this sequence length
        # pos_emb shape: (1, seq_len, embed_dim)
        pos_emb = self.positional_encoding[:t, :].unsqueeze(0)

        # Sum token and positional embeddings then apply dropout
        # (batch_num, seq_len, embed_dim)
        x = self.transformer.drop(tok_emb + pos_emb)

        # Pass through all num_layers transformer decoder blocks
        # Each block: (batch_num, seq_len, embed_dim) → (batch_num, seq_len, embed_dim)
        for block in self.transformer.h:
            x = block(x)

        # Apply final layer normalisation
        # (batch_num, seq_len, embed_dim)
        x = self.transformer.ln_f(x)

        # Project to vocabulary logits
        # (batch_num, seq_len, embed_dim) → (batch_num, seq_len, vocab_size)
        logits = self.lm_head(x)

        # Compute cross-entropy loss if targets are provided
        # logits reshaped: (batch_num * seq_len, vocab_size)
        # targets reshaped: (batch_num * seq_len,)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        return logits, loss

    # ── Autoregressive generation ────────────────────────────────────────
    @torch.no_grad()
    def generate(
        self, idx, max_new_tokens, temperature=1.0, do_sample=False, top_k=None
    ):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b, t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        for _ in range(max_new_tokens):
            # Crop context to block_size if it has grown too long
            # idx_cond: (batch_num, min(current_seq_len, max_len))
            idx_cond = (
                idx if idx.size(1) <= self.block_size else idx[:, -self.block_size :]
            )

            # Run forward pass; take logits at the last time step
            # logits: (batch_num, seq_len, vocab_size)
            logits, _ = self(idx_cond)

            # Extract last-position logits and apply temperature scaling
            # (batch_num, seq_len, vocab_size) → (batch_num, vocab_size)
            logits = logits[:, -1, :] / temperature

            # Convert logits to probabilities (numpy softmax) — (inferred)
            # Shape: (batch_num, vocab_size)
            logits = np.exp(logits) / np.sum(np.exp(logits), axis=-1, keepdims=True)

            # Optionally restrict to top-k tokens
            if top_k is not None:
                # v: (batch_num, top_k)
                v, _ = torch.topk(logits, top_k)
                # Zero-out all logits below the k-th largest
                logits[logits < v[:, [-1]]] = -float("Inf")

            # Normalise to a proper probability distribution
            # Shape: (batch_num, vocab_size)
            probs = F.softmax(logits, dim=-1)

            if do_sample:
                # Sample one token index per batch item
                # (batch_num, vocab_size) → (batch_num, 1)
                idx_next = torch.multinomial(probs, num_samples=1)
            else:
                # Take the argmax token index (greedy decoding)
                # (batch_num, vocab_size) → (batch_num, 1)
                _, idx_next = torch.topk(probs, k=1, dim=-1)

            # Append the new token to the running sequence
            # Input: (batch_num, current_seq_len) + (batch_num, 1)
            # Output: (batch_num, current_seq_len + 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# ── Smoke-test: instantiate, compile, and run one forward pass ───────────
# Create a GPT model and move to device
model = GPT(config).to(config.device)
# Compile model graph for optimised CUDA kernels (requires PyTorch >= 2.0)
model = torch.compile(model)

# Forward pass with sample batch
# x: (batch_num, seq_len)  y: (batch_num, seq_len)
# logits: (batch_num, seq_len, vocab_size)  loss: scalar
logits, loss = model.forward(x, y)

number of parameters: 83.64M


[2023-04-22 02:12:32,380] torch._inductor.utils: [WARNING] using triton random, expect difference from eager


logits:  torch.Size([4, 64, 301966])
loss:  tensor(12.6203, device='cuda:0', grad_fn=<CompiledFunctionBackward>)


# 4) Training

In [ ]:
# ── Trainer: encapsulates the training loop ──────────────────────────────
class Trainer:
    """
    Trainer class to handle the training loop for the GPT model.
    """

    def __init__(self, config, model, train_dataset):
        """
        Initializes the Trainer with model configuration, model, and training dataset.
        Sets up device and moves the model to the specified device.
        Initializes variables for logging and tracking training progress.
        """
        self.config = config
        self.model = model
        self.optimizer = None
        self.train_dataset = train_dataset
        self.callbacks = defaultdict(list)
        self.device = config.device
        # Move model to target device
        self.model = self.model.to(self.device)

        # Training state counters
        self.iter_num = 0
        self.iter_time = 0.0
        self.iter_dt = 0.0

    def add_callback(self, onevent: str, callback):
        """Append a callback to the list for a given training event."""
        self.callbacks[onevent].append(callback)

    def set_callback(self, onevent: str, callback):
        """Replace all callbacks for a given event with a single callback."""
        self.callbacks[onevent] = [callback]

    def trigger_callbacks(self, onevent: str):
        """Invoke all registered callbacks for a given event."""
        for callback in self.callbacks.get(onevent, []):
            callback(self)

    def run(self):
        """
        Main training loop.
        Each iteration: fetch batch → forward pass → backward pass → parameter update.
        Triggers 'on_batch_end' callbacks after each step.
        Stops once max_iters is reached.
        """
        model, config = self.model, self.config

        # Configure AdamW optimiser with separate weight-decay groups
        self.optimizer = model.configure_optimizers(config)

        # Build DataLoader with random infinite sampling
        train_loader = DataLoader(
            self.train_dataset,
            sampler=torch.utils.data.RandomSampler(
                self.train_dataset, replacement=True, num_samples=int(1e10)
            ),
            shuffle=False,
            batch_size=config.batch_size,
            num_workers=config.num_workers,
        )

        model.train()
        self.iter_num = 0
        self.iter_time = time.time()
        data_iter = iter(train_loader)
        while True:

            # Fetch next batch; reset iterator if exhausted
            # x: (batch_num, seq_len)  y: (batch_num, seq_len)
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(train_loader)
                batch = next(data_iter)
            batch = [t.to(self.device) for t in batch]
            x, y = batch

            # Forward pass
            # logits: (batch_num, seq_len, vocab_size)  loss: scalar
            logits, self.loss = model(x, y)

            # Backward pass and parameter update
            model.zero_grad(set_to_none=True)
            self.loss.backward()
            # Clip gradient norms to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_norm_clip)
            self.optimizer.step()

            self.trigger_callbacks("on_batch_end")
            self.iter_num += 1
            tnow = time.time()
            self.iter_dt = tnow - self.iter_time
            self.iter_time = tnow

            # Stop training when maximum iterations reached
            if config.max_iters is not None and self.iter_num >= config.max_iters:
                break


# ── Instantiate model and trainer ────────────────────────────────────────
model = GPT(config)
trainer = Trainer(config, model, train_dataset)
trainer = Trainer(config, model, train_dataset)


# Logging callback: print loss every 500 iterations
def batch_end_callback(trainer):
    """
    Callback function to print training progress at the end of each batch.
    Prints iteration number, iteration time, and training loss.
    """
    if trainer.iter_num % 500 == 0:
        print(
            f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}"
        )


# Register callback and start training
trainer.set_callback("on_batch_end", batch_end_callback)
trainer.run()

number of parameters: 83.64M
iter_dt 0.00ms; iter 0: train loss 12.63450
iter_dt 107.20ms; iter 500: train loss 4.48716
iter_dt 105.59ms; iter 1000: train loss 4.52789
iter_dt 108.12ms; iter 1500: train loss 4.98708
iter_dt 110.25ms; iter 2000: train loss 4.63459
iter_dt 107.60ms; iter 2500: train loss 4.00061
iter_dt 107.23ms; iter 3000: train loss 4.97294
iter_dt 108.26ms; iter 3500: train loss 5.00927
iter_dt 107.98ms; iter 4000: train loss 4.54547
iter_dt 111.64ms; iter 4500: train loss 4.49295
iter_dt 111.27ms; iter 5000: train loss 4.24072
iter_dt 107.57ms; iter 5500: train loss 4.49713
iter_dt 108.68ms; iter 6000: train loss 4.44291
iter_dt 111.30ms; iter 6500: train loss 5.33070
iter_dt 109.34ms; iter 7000: train loss 3.61869
iter_dt 107.99ms; iter 7500: train loss 4.67334
iter_dt 112.82ms; iter 8000: train loss 3.57174
iter_dt 107.69ms; iter 8500: train loss 3.86336
iter_dt 109.40ms; iter 9000: train loss 3.27506
iter_dt 109.84ms; iter 9500: train loss 4.14696
iter_dt 108.70ms

In [ ]:
# ── Autoregressive text generation ──────────────────────────────────────
# Provide a prompt string to condition generation on
text = "Lord:\nRise! My people, conquer the north!"

# Encode prompt text into token IDs and add batch dimension
# sample_ids shape: (seq_len,) → unsqueeze → (batch_num=1, seq_len)
sample_ids = torch.Tensor(enc.encode_ordinary(text)).long()
sample_ids = torch.unsqueeze(sample_ids, 0).to(config.device)

# Run autoregressive generation for max_new_tokens steps
# result shape: (batch_num=1, seq_len + max_new_tokens)
result = model.generate(
    sample_ids, max_new_tokens=50, temperature=1, do_sample=False, top_k=None
)

# Decode generated token IDs back to a human-readable string
# result.detach().cpu().tolist()[0]: list[int] of length (seq_len + max_new_tokens)
print(enc.decode(result.detach().cpu().tolist()[0]))

Lord:
Rise! My people, conquer the north!
And, in the same blood of death,
Is my poor heart-blood, like a very heart,
To see his head in the same fair king's face.

GLOUCESTER:
I'll have his uncle
